# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [14]:
answer = """
One row in the raw data means: one content page (content_hash_id),
for one client (client_hash_id), on one day (report_date).

For modelling, daily observations will be aggregated into historical feature
windows and future target windows, creating one prediction example per page
at a prediction point.
"""
print(answer)


One row in the raw data means: one content page (content_hash_id),
for one client (client_hash_id), on one day (report_date).

For modelling, daily observations will be aggregated into historical feature
windows and future target windows, creating one prediction example per page
at a prediction point.



In [15]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
hf_token = os.environ["HF_TOKEN"]

import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

print("Connected")

Connected


In [16]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=hf_token)

for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [ ]:
rel = "hf://datasets/FlyRank/internship-warehouse"

sample = con.sql(f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    LIMIT 5
""").df()

print(sample.columns.tolist())
sample

In [18]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

print(grain_check.empty)
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

True
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []


In [19]:
counts_check = con.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

print(counts_check)

   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31


In [20]:
availability_check = con.sql(f"""
    SELECT
        client_has_gsc,
        client_has_ga4,
        COUNT(*) as row_count
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY client_has_gsc, client_has_ga4
""").df()

print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   client_has_gsc  client_has_ga4  row_count
0            True           False    3018741
1            True            True    6822637


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [21]:
field_classification = """

FEATURES
(aggregated only from the historical feature window before the prediction point):

- gsc_impressions          - search visibility signal
- gsc_clicks               - search traffic signal
- gsc_avg_position         - average ranking position signal

- ga4_pageviews            - page view signal (when available)
- ga4_sessions             - session signal (when available)
- ga4_users                - user volume signal (when available)
- ga4_engaged_sessions     - engagement signal (when available)
- ga4_total_engagement_sec - engagement duration signal (when available)

- sessions_organic         - organic traffic signal
- sessions_direct          - direct traffic signal
- sessions_referral        - referral traffic signal
- sessions_social          - social traffic signal
- sessions_paid            - paid traffic signal

- sessions_ai              - AI referral traffic signal
- ai_chatgpt               - ChatGPT referral signal
- ai_perplexity            - Perplexity referral signal
- ai_gemini                - Gemini referral signal
- ai_copilot               - Copilot referral signal
- ai_claude                - Claude referral signal
- ai_meta                  - Meta AI referral signal
- ai_other                 - other AI referral signal

- scroll_events            - engagement interaction signal


LABEL / TARGET:

- future_decline_pct_30d:
  Percentage change in the selected performance metric during the
  30-day future target window compared with the historical feature
  window.

The target is calculated only from future-window performance.
Information from the target window will not be included in the
features because that would cause data leakage.


CONTEXT
(used for grouping, joining, window creation, availability checks,
validation, and output):

- report_date              - used to create historical and future windows
- month                    - used for time filtering
- client_hash_id           - used for client grouping and validation splits
- content_hash_id          - used to identify content pages and create output rankings

DATA AVAILABILITY CONTEXT:

- client_has_gsc           - indicates whether the client has GSC data
- client_has_ga4           - indicates whether the client has GA4 data
- gsc_data_available       - indicates whether GSC data is available
- ga4_data_available       - indicates whether GA4 data is available


EXCLUDED:

- gsc_sum_position:
  Excluded because gsc_avg_position provides the average ranking
  position signal used for modelling. The sum of positions is not
  needed as a separate feature.

- Future-window performance metrics:
  Excluded because they are used to calculate the target and would
  reveal information from the future target period.

- Target-derived columns:
  future_decline_pct_30d and any derived decline labels are excluded
  from model features because they represent the outcome being predicted.

- client_hash_id:
  Not used as a model feature because it is an identifier used for
  grouping and validation, not a predictive signal.

- content_hash_id:
  Not used as a model feature because it identifies pages but does not
  contain measurable performance information.

- report_date and month:
  Not used as model features because they define time windows and
  validation, not page performance signals.

"""
print(field_classification)



FEATURES
(aggregated only from the historical feature window before the prediction point):

- gsc_impressions          - search visibility signal
- gsc_clicks               - search traffic signal
- gsc_avg_position         - average ranking position signal

- ga4_pageviews            - page view signal (when available)
- ga4_sessions             - session signal (when available)
- ga4_users                - user volume signal (when available)
- ga4_engaged_sessions     - engagement signal (when available)
- ga4_total_engagement_sec - engagement duration signal (when available)

- sessions_organic         - organic traffic signal
- sessions_direct          - direct traffic signal
- sessions_referral        - referral traffic signal
- sessions_social          - social traffic signal
- sessions_paid            - paid traffic signal

- sessions_ai              - AI referral traffic signal
- ai_chatgpt               - ChatGPT referral signal
- ai_perplexity            - Perplexity referr

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
missing_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(gsc_impressions) AS missing_gsc_impressions,
        COUNT(*) - COUNT(ga4_sessions) AS missing_ga4_sessions,
        COUNT(*) - COUNT(scroll_events) AS missing_scroll_events
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

print(missing_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  missing_gsc_impressions  missing_ga4_sessions  \
0     9841378                        0               3018741   

   missing_scroll_events  
0                3018741  


In [23]:
window_check = con.sql(f"""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS number_of_days
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

print(window_check)

  first_date  last_date  number_of_days
0 2026-03-01 2026-03-31              31


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [24]:
data_limits = """
This data can support measured, directional, and decision-support analysis,
but it cannot tell us everything about why a page's performance changes.

1. Unbalanced history:
   Clients have different amounts of historical data. Some clients have
   much longer performance histories than others, so a single global
   calendar window cannot be assumed to represent the same amount of
   history for every client. Client-specific data availability must be
   checked before creating historical and future windows.

2. GSC-only early history:
   Some clients have GSC data before their GA4 data becomes available.
   During those periods, GA4 fields may be zero-filled while
   ga4_data_available is FALSE. Therefore, a zero GA4 value cannot
   automatically be interpreted as zero engagement or zero traffic.

3. Window overlap and leakage:
   The feature window and future target window must be kept strictly
   separate. Information from the future target period must never be
   included in model features. In particular, the fixed 90-day query
   table overlaps recent months, so its final-month performance fields
   cannot be used as features when that same period defines the target.

4. No causal explanation:
   The data can show associations between historical performance signals
   and future outcomes, but it cannot prove that a particular signal
   caused a page to decline. The model should therefore be treated as
   decision support rather than causal evidence.

5. No guarantee of future performance:
   A model trained on historical patterns can estimate future decline
   risk, but it cannot guarantee what will happen to an individual page.
   External events, search changes, content changes, and other factors
   outside the available data can affect future performance.
"""
print(data_limits)




This data can support measured, directional, and decision-support analysis,
but it cannot tell us everything about why a page's performance changes.

1. Unbalanced history:
   Clients have different amounts of historical data. Some clients have
   much longer performance histories than others, so a single global
   calendar window cannot be assumed to represent the same amount of
   history for every client. Client-specific data availability must be
   checked before creating historical and future windows.

2. GSC-only early history:
   Some clients have GSC data before their GA4 data becomes available.
   During those periods, GA4 fields may be zero-filled while
   ga4_data_available is FALSE. Therefore, a zero GA4 value cannot
   automatically be interpreted as zero engagement or zero traffic.

3. Window overlap and leakage:
   The feature window and future target window must be kept strictly
   separate. Information from the future target period must never be
   included in model f

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.